[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astudentfromsustech/agents-course/blob/main/practice/p2_vit_mnist/vit_mnist.ipynb)

# p2 · A transformer that reads handwritten digits

An image is not a sequence. A **Vision Transformer** makes it one: cut the 28×28 image
into 7×7 patches, flatten each into a vector, and call that the sequence — 16 tokens,
plus a `[CLS]` token whose final vector the classifier reads. Everything after that is
chapter 01 unchanged: Q/K/V, softmax, multi-head, residual, norm.

Three things the chapter *claims*, measured here instead of asserted:

| | what it shows |
|---|---|
| `train` | it works, and the attention maps are readable |
| `position` | attention is permutation-equivariant — strip the positional encoding and shuffling the patches changes **nothing** |
| `scale` | a CNN's locality is a free prior, worth most when data is scarce |

Runs on a Mac GPU (MPS), a Colab GPU, or plain CPU — the next cell picks whichever it finds.

In [ ]:
# Setup. Works on Colab and on your own machine; no torchvision needed.
import os, sys, importlib, urllib.request

URL = ("https://raw.githubusercontent.com/astudentfromsustech/agents-course"
       "/main/practice/p2_vit_mnist/vit_mnist.py")
IN_COLAB = "google.colab" in sys.modules

# On Colab always re-fetch, so a restarted session cannot keep running stale code.
# Locally never overwrite: that file is yours to edit.
if IN_COLAB or not os.path.exists("vit_mnist.py"):
    urllib.request.urlretrieve(URL, "vit_mnist.py")

import vit_mnist as V
importlib.reload(V)      # a warm kernel holds the OLD module even after a fresh download

print("device:", V.DEV)  # mps = your Mac's GPU | cuda = Colab GPU | cpu = fallback
print("has show_data:", hasattr(V, "show_data"))

## The data, and what a patch is

MNIST arrives as one 11 MB `.npz` — 60 000 training images, 10 000 test.
`PatchEmbed` is a `Conv2d` whose kernel equals its stride, so it does **no mixing at
all**: it only reshapes pixels into tokens. Every bit of spatial reasoning after that
point is done by attention.

In [ ]:
d = V.load()
(Xtr, Ytr), (Xte, Yte) = d
print("train", tuple(Xtr.shape), " test", tuple(Xte.shape))

m = V.ViT().to(V.DEV)
print("tokens per image:", V.NP, "patches +1 CLS =", V.NP + 1)
print("params:", f"{sum(p.numel() for p in m.parameters()):,}")
print("one image -> patch sequence:", tuple(m.patch(Xte[:1]).shape))   # [1, 16, 64]

# ...and actually look at it
from IPython.display import Image, display
display(Image(V.show_data(d)))

**Read the bottom strip.** That row of 16 squares is the entire input — the model never
receives the 4x4 arrangement, only those tokens in order. Token 5 and 6 hold the top bar of
the `7`, 10 and 13 the diagonal.

Which is the setup for experiment 2: if you shuffled that strip, would the model notice?

### What is actually in those 16 x 64 numbers?

"16 tokens x 64" is abstract until you look at it. Each **row** is one token — 64 numbers
describing one 7x7 patch.

In [ ]:
from IPython.display import Image, display
display(Image(V.show_tokens(d)))

Read the heatmap row by row and it maps onto the digit: rows **5, 6** are the top bar of the
`7`, rows **10, 13** the diagonal. Those carry strong values. The pale rows outlined in green —
**0-3, 8, 11, 12, 15** — are the blank patches, and they are *byte-identical to each other*.

Not approximately. Exactly. A blank patch is 49 copies of the same background value, so its
token is `W @ (constant) + bias`, the same arithmetic every time, wherever the patch sat.

**Nothing about position has entered yet.** Token 0 (top-left) and token 15 (bottom-right) are
indistinguishable here. The positional encoding one step later is the only thing that will ever
tell them apart — which is precisely why experiment 2 comes out the way it does.

Note these values come from an **untrained** model, so `W` is random and the numbers carry no
learned meaning. What survives training is the structure: identical patches give identical
tokens, inked patches give large-magnitude vectors, and the 2-D layout contributes nothing.

### Why 64, when a patch is only 7x7 = 49 pixels?

The two numbers come from different places. **49 is forced** — it is what you get by cutting
the image into 7x7 squares. **64 is chosen**: it is `d_model`, picked by whoever designed the
model, and nothing derives it from 49.

So what is in the extra 15 dimensions? Measure it.

In [ ]:
V.why_64(d);

**Nothing, yet.** The embedding matrix is `64 x 49`, so its rank is at most 49 — and the
singular values of 64,000 real tokens fall off a cliff at exactly index 49 (a 200,000x drop).
Every token lives in a **49-dimensional slice** of the 64-d space, offset by the bias. The
other 15 directions are empty.

Which is the point: `d_model` is the width of the **residual stream for the whole network**,
not just of the patch encoder. It has to carry everything the model accumulates about a
position across all four blocks — what this patch looks like, *where* it sits, what it read
from the other 16 tokens, what `[CLS]` should take from it. The 49 pixel-derived numbers are
only the starting content; the spare directions are scratch space the blocks write into,
because every sub-layer **adds** to the stream rather than replacing it.

Three things actually constrain the choice of 64:

| constraint | why |
|---|---|
| **divisible by the head count** | 64 / 4 heads = 16 each. You could not use 49 with 4 heads at all |
| **hardware alignment** | GPU matmuls tile in multiples of 8/16/64; 49 would be padded anyway |
| **big enough not to collide** | `d_model = 32` would make `49 -> 32` non-injective: different patches forced onto the same vector |

For scale, real ViTs sit near equality rather than widening: ViT-Base/16 uses 16x16x3 = **768**
pixels per patch and `d_model` = **768**. Our 49 -> 64 is a mild expansion by comparison.

> A measurement caveat worth copying: the first time I checked this I used
> `torch.linalg.matrix_rank` with an absolute tolerance and got "rank 64", which is wrong — it
> counted float32 noise as real directions. The singular-value spectrum is the honest view, and
> a rank number from a default tolerance is not.

## The whole model in one picture

Drawn the way CNN papers draw an architecture — left to right, each slab a tensor, the digit at
one end and the ten classes at the other.

In [ ]:
from IPython.display import Image, display
display(Image(V.draw_architecture(d)))

Compare it with the CNN diagram it imitates. A CNN's slabs **shrink** — 28x28 to 24x24 to
12x12 to 8x8 to 4x4 — because each convolution and pooling step throws away spatial resolution
to buy channels. The ViT's slab is `17 x 64` at the start of the stack and `17 x 64` at the
end: **nothing shrinks, ever**. There is no pooling and no stride.

That is the structural difference between the two families. A CNN builds a pyramid; a
transformer keeps a fixed-size workspace and rewrites it in place, 4 times here, ~100 times in
a large language model.

### One image, or a batch?

The diagram is drawn for **one** image, the way architecture diagrams always are. Real training
is **batched**: 256 images go through together, so every tensor carries a leading batch axis —
`(256, 17, 64)`, not `(17, 64)`.

Two reasons to batch, and they are different:

- **speed** — one matmul over 256 images saturates a GPU; 256 separate matmuls do not
- **a less noisy gradient** — the slope from a single example is a poor estimate of the slope
  over the dataset; averaging 256 of them is much steadier

Which raises the obvious worry: *if 256 images share a tensor, do they contaminate each other?*
Don't assume — measure.

In [ ]:
V.batch_check(d);

**No.** Image 0's logits are bit-for-bit the same whether it travels with 255 companions or
entirely alone (the `1e-07` is floating-point noise, not interaction).

The reason is in the shape: the attention matrix is `(256, 4, 17, 17)` — **256 separate 17x17
matrices**, one per image per head. The batch axis is carried along by every operation and
mixed by none of them. `q @ k.transpose(-2,-1)` contracts the *last* dimension, so a token of
image *i* is never scored against a token of image *j*.

The one place the batch really does combine is the **loss**: `cross_entropy` averages over all
256, which is exactly the point — one averaged gradient, applied once.

## Where the data goes, shape by shape

Before the loop, follow one batch all the way through. Every shape below comes from a **live
tensor** produced by the same modules training uses — nothing is written by hand, so it cannot
drift from the code.

In [ ]:
V.trace_shapes(d);

In [ ]:
# the same trace, drawn
from IPython.display import Image, display
display(Image(V.draw_flow()))

Four things to notice.

**The image becomes a sequence once, at the start.** `Conv2d` gives `(256, 64, 4, 4)` — still
laid out as a 4x4 grid — and `flatten(2).transpose(1,2)` turns that into `(256, 16, 64)`.
After that line the 2-D layout is *gone*; only the positional encoding remembers it.

**The 17x17 is the attention matrix.** 17 tokens each scoring all 17, per head, per image:
`(256, 4, 17, 17)`. That is the only place information crosses between positions. Everything
else — LayerNorm, the MLP, every Linear — works on one token at a time.

**The shape never changes inside the stack.** In `(256, 17, 64)`, out `(256, 17, 64)`, four
times. That invariance is exactly why blocks stack: each one *adds* to a running total rather
than transforming it into something a different shape.

**Only `[CLS]` survives.** `t[:, 0]` keeps row 0 and discards the other 16 tokens. They were
not wasted — attention let `[CLS]` read them for four layers — but the classifier sees one
64-vector per image, which becomes 10 logits, which become one loss number.

## How the training loop works

`exp_train` hides it, so before running it, look inside. Five lines do all the work, once per
batch:

```python
logits = model(x)                  # guess
loss   = cross_entropy(logits, y)  # how wrong, as ONE number
opt.zero_grad()                    # clear last batch's gradients
loss.backward()                    # d(loss)/d(every one of 204,298 parameters)
opt.step()                         # nudge each parameter downhill
```

"6 epochs" is really **1,410 updates**: 60 000 images ÷ 256 per batch = 235 batches, six times
over. An *epoch* is one pass over the data; a *step* is one parameter update.

In [ ]:
from IPython.display import Image, display
display(Image(V.show_training(d)))        # retrains while recording every step, ~15 s

**Why the printed loss jitters** — it is just the *last* batch's loss, one of 235, on 256 random
images. Some batches are harder than others. The blue moving average is the real trend.

**What "loss" means** — cross-entropy is `-log(probability given to the true class)`. At init the
model spreads probability evenly over the 10 classes, so `p(true) ≈ 0.1` and
`-log(0.1) ≈ 2.3`, which is exactly where the curve starts. The bottom-right panel shows one
digit going from 0.077 to 0.998.

Loss and accuracy are **not** the same thing: accuracy only asks whether the top bar is right,
loss cares how confident it was. That is why the orange curve keeps climbing after the blue one
flattens.

## 1 · Train it

~11 s on an M-series Mac. Watch the test accuracy climb.

In [ ]:
m = V.exp_train(d)

`attention_maps.png` is written next to the notebook. The bottom row is the `[CLS]`
token's attention over the 4×4 patch grid — **not uniform**: on a `1` the mass runs up the
stroke, on a `0` it sits around the ring.

Attention weights are one of the few interpretable things in a neural network, because
they are a distribution over *inputs* rather than over hidden features.

In [ ]:
from IPython.display import Image, display
display(Image("attention_maps.png"))

## 2 · Position: the one thing attention cannot see

Train two identical models, one with the positional encoding switched off. Then **shuffle
the patch order at test time**.

Predict the four numbers before you run it.

In [ ]:
rows = V.exp_position(d)

The `+0.00` is the whole point. The model with no positional encoding is not merely
*robust* to the shuffle — it is **mathematically incapable of noticing it**, because the
only thing it ever received was a set. That is permutation equivariance, measured.

The gap in the first column is what position buys you.

## 3 · Scale: what a prior is worth

Same parameter budget, same number of gradient steps, growing training set. A CNN has
locality *hard-coded*; the transformer has to learn it.

In [ ]:
out = V.exp_scale(d)

**Read the gap column, not the accuracy columns.** The CNN's locality prior is free
knowledge handed over before training starts, and it is worth about **19 points at
N = 1 000 and 3 points at N = 60 000**.

No crossover here — the ViT never overtakes the CNN on MNIST, because all of MNIST sits
inside the small-data regime (the crossover in the ViT paper is somewhere past 100 M
images). What a laptop can show you is the **slope**: the prior paying off less and less.
That slope is the whole argument of the chapter's `Self-attention vs CNN` section.

## Things worth trying

- `V.PATCH = 4` (49 tokens) or `14` (4 tokens) — where does accuracy break, and is the cost
  compute or information?
- `V.LAYERS = 1` — how much does one round of mixing lose?
- Shuffle the **pixels inside** each patch instead of the patches. Why does that hurt the
  no-position model, when shuffling patches did not?
- `PatchEmbed` is a `Conv2d` with `kernel_size == stride`. Make them differ so patches
  overlap — you have just reintroduced a locality prior. Does the N = 1 000 number move
  toward the CNN's?